# 11.2 — One-at-a-time sensitivity

**Question.** Which preserved historical configuration parameters move selected-policy outcomes beyond baseline optimizer noise? The `test` profile runs one parameter grid on synthetic cells; larger profiles use the prespecified screening/full grids and env-resolved features. The exact count is previewed before execution.

The shared runner resumes signed artifacts under the legacy sensitivity root. Response curves are conditional OAT diagnostics: they do not capture parameter interactions, establish causal effects, or convert dimensionless model settings into empirical uncertainty. Run 11.1 at the same output root first so effect-to-noise ratios use fixed-default seed variation.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from estonia_landuse.sensitivity.analysis import summarize_oat
from estonia_landuse.sensitivity.config import DEFAULT_SEEDS, OAT_PARAMETERS
from estonia_landuse.sensitivity.plots import plot_oat_response_curves
from estonia_landuse.sensitivity.runner import run_manifest
from estonia_landuse.sensitivity.sampling import build_oat_manifest, manifest_run_count, manifest_summary

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
PROFILE = os.environ.get("SENSITIVITY_PROFILE", "test")
N_WORKERS = int(os.environ.get("SENSITIVITY_N_WORKERS", "2"))
OVERWRITE = os.environ.get("SENSITIVITY_OVERWRITE", "false").lower() == "true"
OUTPUT_ROOT = Path(os.environ.get("SENSITIVITY_OUTPUT_ROOT", PROJECT_ROOT / "data/processed/legacy_sensitivity")).resolve()
FEATURES_PATH = Path(os.environ.get("SENSITIVITY_FEATURES_PATH", PROJECT_ROOT / "data/processed/learned_carbon/features_with_forest.parquet")).resolve()
SEEDS = (0, 1) if PROFILE == "test" else DEFAULT_SEEDS[PROFILE]
SCENARIOS = ("balanced",)
OUTCOMES = ("biodiversity_gain", "carbon_gain", "cost", "changed_pct")


In [ ]:
if PROFILE == "test":
    position = np.linspace(0.0, 1.0, 12)
    context = pd.DataFrame({"cell_id": np.arange(1, 13), "forest_pct": 0.35 + 0.03 * position, "wetland_pct": 0.10 + 0.02 * position, "agriculture_pct": 0.30 - 0.03 * position, "grassland_pct": 0.15 - 0.02 * position, "urban_pct": np.full(12, 0.05), "water_pct": np.full(12, 0.05), "protected_overlap_pct": 0.05 * position, "wetland_suitability": 0.2 + 0.6 * position, "opportunity_cost_proxy": 0.1 + 0.5 * position, "predicted_tco2_ha_yr": 2.5 + 2.0 * position, "peat_overlap_pct": 0.4 * position})
    feature_columns = ["wetland_suitability", "opportunity_cost_proxy"]
else:
    if not FEATURES_PATH.exists(): raise FileNotFoundError(f"Missing historical feature input: {FEATURES_PATH}")
    context = pd.read_parquet(FEATURES_PATH)
    feature_columns = [name for name in ("urban_pct", "agriculture_pct", "grassland_pct", "forest_pct", "wetland_pct", "water_pct", "naturalness_score", "carbon_score", "protected_overlap_pct", "wetland_suitability", "biodiversity_proxy", "opportunity_cost_proxy", "rohemeeter_norm") if name in context]
    if not feature_columns: raise ValueError("No preserved Notebook 10 feature columns found")


In [ ]:
manifest = build_oat_manifest(profile=PROFILE, scenarios=SCENARIOS, seeds=SEEDS)
if PROFILE == "test":
    manifest = manifest.loc[manifest["parameter"].eq(next(iter(OAT_PARAMETERS)))].reset_index(drop=True)
planned_runs = manifest_run_count(manifest)
manifest_summary(manifest)
display(manifest.head(6))


In [ ]:
statuses = run_manifest(context, feature_columns, manifest, OUTPUT_ROOT, PROFILE, overwrite=OVERWRITE, n_workers=min(N_WORKERS, planned_runs), progress=lambda completed, total, status: print(f"[{completed}/{total}] {status}"))
if statuses["status"].eq("failed").any(): raise RuntimeError(statuses.loc[statuses["status"].eq("failed"), ["sample_id", "seed", "error_message"]].to_string(index=False))
assert len(statuses) == planned_runs
display(statuses["status"].value_counts())


In [ ]:
metrics = pd.concat([pd.read_parquet(path) for path in statuses["metrics_path"]], ignore_index=True)
baseline_paths = sorted((OUTPUT_ROOT / "runs" / "baseline").rglob("seed_*.parquet"))
if not baseline_paths: raise FileNotFoundError("Run Notebook 11.1 with this SENSITIVITY_OUTPUT_ROOT first")
baseline = pd.concat([pd.read_parquet(path) for path in baseline_paths], ignore_index=True)
curves, effect_to_noise = summarize_oat(metrics, baseline, OUTCOMES)
display(effect_to_noise.sort_values("effect_to_noise", ascending=False))
for outcome in OUTCOMES:
    figure, _ = plot_oat_response_curves(curves, outcome)
    display(figure)
    plt.close(figure)
